# Nike Ecommerce Agent

In [ ]:
from dotenv import load_dotenv
from agent import RunConfig,OpenAIChatCompletionsModel,set_tracing_disabled
from openai import AsyncOpenAI
import os
# Load environment variables
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if GOOGLE_API_KEY is None:
    raise ValueError("GOOGLE_API_KEY environment variable is not set.")

# Set up the Gemini API-compatible client
client = AsyncOpenAI(
    api_key=GOOGLE_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

model = OpenAIChatCompletionsModel(
    model="gemini-2.0-flash",
    openai_client=client
)

# Wrap into RunConfig
config = RunConfig(
    model=model,
    model_provider=client,
    tracing_disabled=True,
)

set_tracing_disabled(True)

import nest_asyncio
nest_asyncio.apply()


In [ ]:
from agent import Agent, Runner, TResponseInputItem
async def start_chat(primary_agent: Agent, chat: list[TResponseInputItem]):
   
    print("NOTE: Chat started. You can type 'EXIT' to exit the conversation.")
    print("-----------------------------------------")
    while True:

        user_input = input("You: ")
        print("User: ", user_input, "\n")
        
        if user_input == "EXIT":
            print("Fitness Assistant: Goodbye!", "\n")
            break
        
        chat.append({
            "content": user_input,
            "role" : "user",
            "type": "message"
        })
        
        result = await Runner.run(
            starting_agent=primary_agent, 
            input=chat
        )
        
        chat.clear()
        chat.extend(result.to_input_list())


        print("Nike Assistant:", result.final_output, "\n", flush=True)

In [27]:
from dataclasses import dataclass
from typing import List

@dataclass
class UserInfo:
    name: str
    age: int
    email: str

@dataclass
class ProductEstimation:
    product_name: str
    product_type: str
    price_range: int

@dataclass
class ProductRecommendation:
    product_name: str
    product_type: str
    product_price: int
    product_slug: str

@dataclass
class OrderDetail:
    order_status: str
    estimated_delivery_date: str
    no_of_products: int
    total_price: int

@dataclass
class OrderTracker:
    total_orders: int
    orders_detail: List[OrderDetail]

@dataclass
class UserProfile:
    user_info: UserInfo
    product_estimation: ProductEstimation
    product_recommendation: ProductRecommendation
    order_tracker: OrderTracker


In [ ]:
import json
from agent import Agent, FunctionTool, RunContextWrapper, function_tool, Runner, TResponseInputItem
import pandas as pd
import requests

@function_tool
async def get_user_profile(wrapper:RunContextWrapper[UserProfile]):
    """
    Get user profile information.
    """
    try:
        return("User Profile: ", wrapper.context.model_dump_json())
    except Exception as e:
        print(f"Failed to view user profile: {e}")
        raise

@function_tool
async def save_product_estimation(wrapper:RunContextWrapper[UserProfile],product_name:str, product_type:str, price_range:int):
    """
    Save product estimation information of user.
    Args:
        product_name (str): Name of the product.
        product_type (str): Type of the product.
        price_range (int): Price range of the product.
    Returns:
        str: Confirmation message.
    Raises:
        Exception: If there was an error saving the product estimation plan
    """
    try:
        wrapper.context.product_estimation = ProductEstimation(
            product_name=product_name,
            product_type=product_type,
            product_price=price_range
        )
        return("Product Estimation Saved: ", wrapper.context.product_estimation.model_dump_json())
    except Exception as e:
        print(f"Failed to save product estimation: {e}")
        raise

@function_tool
async def save_product_recommendation(wrapper:RunContextWrapper[UserProfile],product_name:str, product_type:str, product_price:int, product_link:str):
    """
    Save product recommendation information of user.
    Args:
        product_name (str): Name of the product.
        product_type (str): Type of the product.
        product_price (int): Price of the product.
        product_link (str): Link to the product.
    Returns:
        str: Confirmation message.
    Raises:
        Exception: If there was an error saving the product recommendation
    """
    try:
        wrapper.context.product_recommendation = ProductRecommendation(
            product_name=product_name,
            product_type=product_type,
            product_price=product_price,
            product_link=product_link
        )
        return("Product Recommendation Saved: ", wrapper.context.product_recommendation.model_dump_json())
    except Exception as e:
        print(f"Failed to save product recommendation: {e}")
        raise

@function_tool
async def save_order_tracker(wrapper:RunContextWrapper[UserProfile],total_orders:int, orders_detail:List[OrderDetail]):
    """
    Save order tracker information of user.
    Args:
        total_orders (int): Total number of orders.
        orders_detail (List[OrderDetail]): List of order details.
    Returns:
        str: Confirmation message.
    Raises:
        Exception: If there was an error saving the order tracker
    """
    try:
        wrapper.context.order_tracker = OrderTracker(
            total_orders=total_orders,
            orders_detail=orders_detail
        )
        return("Order Tracker Saved: ", wrapper.context.order_tracker.model_dump_json())
    except Exception as e:
        print(f"Failed to save order tracker: {e}")
        raise

@function_tool
async def get_order_history(wrapper:RunContextWrapper[UserProfile], user_email: str) -> str:
    """
    Retrieves order history data for a given user email from an API and returns it as a Pandas DataFrame string.

    Args:
        user_email: The email address of the user.

    Returns:
        A string representation of a Pandas DataFrame containing order details
        (id, order_date, product_name, price, quantity, sub_total, paymentStatus, orderStatus).
        Returns an error message as a string if there are issues with the API request or data processing.
    """
    try:
        response = await requests.get(f"http://localhost:3000/api/users?email={user_email}")
        response.raise_for_status()  # Raise an exception for bad status codes

        data = await response.json()['data']
        data_frame = {
            "order_date": [],
            "product_name": [],
            "price": [],
            "quantity": [],
            "sub_total": [],
            "paymentStatus": [],
            "orderStatus": []
        }
        for order in data.get('orderHistory', []):  # Use .get() to handle missing key
            for item in order.get('productDetails', []):  # Use .get() to handle missing key
                data_frame['order_date'].append(pd.to_datetime(order.get('orderDate')).date())
                data_frame['product_name'].append(item['product_id'].get('productName'))
                data_frame['price'].append(item['product_id'].get('price'))
                data_frame['quantity'].append(item.get('quantity'))
                data_frame['sub_total'].append(item.get('subtotal'))
                data_frame['paymentStatus'].append(order.get('paymentStatus'))
                data_frame['orderStatus'].append(order.get('orderStatus'))

        df = pd.DataFrame(data_frame)
        return str(df)

    except requests.exceptions.RequestException as e:
        return f"Error: API request failed - {e}"
    except (KeyError, ValueError) as e:
        return f"Error: Could not process API response - {e}"
    except Exception as e:
        return f"Error: An unexpected error occurred - {e}"

@function_tool()
async def get_product_data(wrapper:RunContextWrapper[UserProfile]):
    """
    Fetches product data from an API and returns it as a Pandas DataFrame.
    """
    response = await requests.get("https://nike-marketplace-bi-structure.vercel.app/api/products")
    data = await response.json()
    products = []
    for item in data["data"]:
        products.append({
            "Product Name": item.get("productName", ""),
            "Price": item.get("price", ""),
            "Inventory": item.get("inventory", ""),
            "Colors": ", ".join(item.get("colors", [])),  # Convert list to comma-separated string
            "Status": item.get("status", ""),
            "Category": item.get("category", ""),
            "Slug": item.get("slug", {}).get("current", "")
        })

    # Convert to DataFrame
    df = pd.DataFrame(products)
    return df

@function_tool
async def save_user_info(wrapper:RunContextWrapper[UserProfile], name:str, age:int, email:str):
    """
    Save user information.

    Args:
        name (str): Name of the user.
        age (int): Age of the user.
        email (str): Email of the user.
    Returns:
        str: Confirmation message.
    """
    try:
        wrapper.context.user_info = UserInfo(
            name=name,
            age=age,
            email=email
        )
        return("User Information Saved: ", wrapper.context.user_info.model_dump_json())
    except Exception as e:
        print(f"Failed to save user information: {e}")
        raise

@function_tool
async def get_review_data(wrapper:RunContextWrapper[UserProfile],product_name: str) -> str:
    """
    Retrieves review data (comment and rating) for a given product name from an API.

    Args:
        product_name: The name of the product to fetch reviews for.

    Returns:
        A string representation of a Pandas DataFrame containing 'comment' and 'rating'
        if reviews are found. Returns "No review and rating available" if no reviews
        are found, or "Error: Product not found" if the API request fails.
    """
    try:
        list_of_words = product_name.split(" ")
        lower_list = [part.lower() for part in list_of_words]
        slug = "-".join(lower_list)
        slug = slug.replace("'", "")

        response = requests.get(f"https://nike-marketplace-bi-structure.vercel.app/api/products?slug={slug}")
        response.raise_for_status()  # Raise an exception for bad status codes

        data = response.json()
        if "data" in data and "reviews" in data["data"] and data["data"]["reviews"]:
            reviews_data = data["data"]["reviews"]
            df_reviews = pd.DataFrame(reviews_data)
            df_review_rating = df_reviews[["comment", "rating"]]
            return str(df_review_rating)
        else:
            return "No review and rating available"

    except requests.exceptions.RequestException as e:
        return f"Error: API request failed - {e}"
    except (KeyError, ValueError):
        return "Error: Invalid API response format"

In [ ]:
from agent import handoff
from agents.extensions import handoff_filters

MainAgent = Agent(
    name="Nike Assistant",
    model=model,
    instructions="Nike Assistant is a virtual assistant that helps users with product recommendations, finding product, order tracking, and user profile management by using different agents. Don't do anything by yourself. Just handover to the respective agent.",
    tools=[
        get_user_profile,
        save_product_estimation,
        save_product_recommendation,
        save_order_tracker,
        get_order_history,
        get_product_data,
        save_user_info
    ]
)



# Product Finder Agent
ProductFinderAgent = Agent(
    name="Product Finder",
    instructions="You are a product finder agent. You help users find products based on their preferences.First get the information about the product save it in ProductEstimation using save_product_estimation and then find the product using tool get_product_data. and then provide the product details. After that handover back to the Nike Assistant to get the product recommendation.",
    model=model,
    handoffs=[handoff(
        agent=MainAgent,
        input_filter=handoff_filters.remove_all_tools
    )],
    tools=[
        get_product_data,
        save_product_estimation,
        save_product_recommendation,
        save_order_tracker,
        get_order_history, 
    ]
)

# Product Recommendation Agent
ProductRecommendationAgent = Agent(
    name="Product Recommendation",
    instructions="You are a product recommendation agent. You help users to suggest the product based on their preferences (Product Estimation) and then save it in ProductRecommendation using save_product_recommendation . then handoff to the main agent.",
    model=model,
    handoffs=[handoff(
        agent=MainAgent,
        input_filter=handoff_filters.remove_all_tools
    )],
    tools=[
        get_product_data,
        save_product_estimation,
        save_product_recommendation,
        save_order_tracker,
        get_order_history, 
    ]
)

# Order Tracker Agent
OrderTrackerAgent = Agent(
    name="Order Tracker",
    instructions="You are an order tracker agent. You help users to track their orders. First get the user email and then get the order history using tool get_order_history and save it in OrderTracker using save_order_tracker. After that handover back to the Nike Assistant.",
    model=model,
    handoffs=[handoff(
        agent=MainAgent,
        input_filter=handoff_filters.remove_all_tools
    )],
    tools=[
        get_order_history,
        save_order_tracker, 
    ]
)


#  Review Analysis Agent
ReviewAnalysisAgent = Agent(
    name="Review Analysis",
    instructions="You are a review analysis agent. You help users to get the reviews and ratings of the product. First get the product name and then get the review data using tool get_review_data.then handover back to the Nike Assistant.",
    model=model,
    handoffs=[handoff(
        agent=MainAgent,
        input_filter=handoff_filters.remove_all_tools
    )],
    tools=[
        get_review_data
    ]
)

# Support Chat Agent
SupportChatAgent = Agent(
    name="Support Chat",
    model=model,
    instructions="""You are a support chat agent. You help users about faqs of company.  After that handover back to the Nike Assistant.
    
    Common FAQs:
    1. What is the return policy?
    2. Where this is locally available?
    3. What payment methods are accepted?
    4. How do I contact customer service?
    5. What is the warranty policy?

    Common Responses:
    1. Our return policy allows you to return items within 30 days of purchase.
    2. You can find our products at only website.
    3. Now we accept only cash on delivery. But we are working on it to add more payment methods.
    4. You can contact customer service by email bistructure9211@gmail.com
    5. Our warranty policy covers defects in materials and workmanship for 1 month from the date of purchase.

    
    """,
    handoffs=[handoff(
        agent=MainAgent,
        input_filter=handoff_filters.remove_all_tools
    )]
)



In [33]:
MainAgent.handoffs = [
    handoff(OrderTrackerAgent),
    handoff(ProductFinderAgent),
    handoff(ProductRecommendationAgent),
    handoff(ReviewAnalysisAgent),
]

In [34]:
chat = []
result = await start_chat(MainAgent, chat)

NOTE: Chat started. You can type 'EXIT' to exit the conversation.
-----------------------------------------
User:  Hi  

Nike Assistant: Hi there! How can I help you today?
 

User:  I want to buy some shoes 

Nike Assistant: Okay, I can help you with that. First, what kind of shoes are you looking for? And what is your price range?
 

User:  I am looking for men's shoes and my price range is 10000 to 15000 

Failed to save product estimation: ProductEstimation.__init__() got an unexpected keyword argument 'product_price'
Failed to save product estimation: ProductEstimation.__init__() got an unexpected keyword argument 'product_price'
Nike Assistant: I am transferring you to the Nike Assistant to get the product recommendation.
 

User:  Get the user profile data 

Failed to view user profile: 'NoneType' object has no attribute 'model_dump_json'
Nike Assistant: I am sorry, I am having some trouble getting the user profile data. Please try again.
 

User:  EXIT 

Fitness Assistant: Good